In [0]:
import dlt
from pyspark.sql.functions import col, to_timestamp, current_timestamp, regexp_replace,regexp_extract,trim
from pyspark.sql.types import StringType, TimestampType

In [0]:
@dlt.view(
    name = "cleaned_articles",
    comment = "Cleansed and enriched articles from bronze layer"
)
@dlt.expect_or_drop("valid_url", "url is not null")
@dlt.expect_or_drop("valid_title", "title is not null")
def cleaned_articles():
    return (
        dlt.read_stream("news_pipeline.bronze.raw_articles")
        .withColumn("publishedAt", to_timestamp(col("publishedAt"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
        .withColumn("content", regexp_replace(col("content"), "<[^>]+>", ""))
        .withColumn("content",regexp_replace(col("content"),r"\[\+\d+ chars\]",""))
        .withColumn("content", regexp_replace(col("content"), "&amp;", "&"))
        .withColumn("title", trim(col("title")))
        .withColumn("source_id", col("source.id"))
        .withColumn("source_name", col("source.name"))
        .withColumn("ingest_time", current_timestamp())
        .withColumn("author",regexp_extract(col("author"),r"\(([^)]+)\)",1))
        .drop("source")
                                               
)

In [0]:
@dlt.create_streaming_table(
    name = "silver_articles",
    comment = "Cleasned and deduplicated silver articles table, with CDC logic applied"
)

@dlt.apply_changes(
    target="silver_articles",
    source="cleaned_articles",
    keys=["url"],
    sequence_by=col("ingest_time")
)